In [ ]:
from pathlib import Path
import os
import sys

INPUT_ROOT = Path('/kaggle/input')
candidates = sorted({path.parent for path in INPUT_ROOT.rglob('requirements.txt') if path.parent.name == 'ner'}, key=str) if INPUT_ROOT.is_dir() else []
NER_DIR = candidates[0] if candidates else Path(r'D:\quantum_task\src\ner')
SRC_DIR = NER_DIR.parent
PROJECT_DIR = SRC_DIR.parent
data_candidates = sorted([path.parent for path in INPUT_ROOT.rglob('few_nerd_mountains_output') if path.is_dir()], key=str) if INPUT_ROOT.is_dir() else []
DATA_ROOT = data_candidates[0] if data_candidates else NER_DIR / 'data'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else PROJECT_DIR

print('PROJECT_DIR =', PROJECT_DIR)
print('SRC_DIR =', SRC_DIR)
print('NER_DIR =', NER_DIR)
print('WORK_DIR =', WORK_DIR)


In [ ]:
assert NER_DIR.exists(), f'Not found: {NER_DIR}'

os.chdir(WORK_DIR)

for path in (PROJECT_DIR, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ['PYTHONPATH'] = os.pathsep.join([str(PROJECT_DIR), str(SRC_DIR)])

print('cwd =', Path.cwd())
print('PYTHONPATH =', os.environ['PYTHONPATH'])

In [ ]:
!ls -la {NER_DIR}
!python --version
!python -c "import sys; sys.path.insert(0, '{PROJECT_DIR}'); sys.path.insert(0, '{SRC_DIR}'); import ner; print('import ok')"

In [ ]:
!python -m pip install -q -r {NER_DIR / 'requirements.txt'}


## NER scripts

### Pretrained bert inference

In [ ]:
%%time 


!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'evaluation' / 'span_marker_benchmark.py'} \
  --data-path {DATA_ROOT / 'few_nerd_mountains_output'} \
  --tuning-split validation \
  --evaluation-split test \
  --model tomaarsen/span-marker-roberta-large-fewnerd-fine-super \
  --label-name location-mountain \
  --batch-size 16 \
  --n-trials 20 \
  --threshold-low 0.3 \
  --threshold-high 0.95 \
  --output-path /kaggle/working/span_marker_fewnerd_results.csv

In [ ]:
%%time

!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'evaluation' / 'gliner_benchmark.py'} \
  --data-path {DATA_ROOT / 'few_nerd_mountains_output'} \
  --model urchade/gliner_large-v2.1 \
  --model urchade/gliner_medium-v2.1 \
  --model urchade/gliner_small-v2.1 \
  --label-text mountain \
  --tuning-split validation \
  --evaluation-split test \
  --n-trials 20 \
  --threshold-low 0.05 \
  --threshold-high 0.95 \
  --output-path /kaggle/working/gliner_results.json

### training ce loss Bert


In [ ]:
%%time

!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'fine_tuning' / 'train_bert.py'} \
  --loss-name cross_entropy \
  --model-name bert-base-cased \
  --train-batch-size 16 \
  --learning-rate 2e-5 \
  --num-epochs 30 \
  --max-length 256 \
  --gradient-accumulation-steps 1 \
  --device cuda \
  --fp16 \
  --output-dir /kaggle/working/bert_base_ce_results \
  --save-model-path /kaggle/working/bert_base_ce_model \
  --disable-wandb

### training weighted ce Bert

In [ ]:
%%time

!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'fine_tuning' / 'train_bert.py'} \
  --loss-name weighted_cross_entropy \
  --model-name bert-base-cased \
  --train-batch-size 16 \
  --learning-rate 2e-5 \
  --num-epochs 30 \
  --max-length 256 \
  --gradient-accumulation-steps 1 \
  --device cuda \
  --fp16 \
  --output-dir /kaggle/working/bert_base_wce_results \
  --save-model-path /kaggle/working/bert_base_wce_model \
  --disable-wandb

### ce Bert evaluation

In [ ]:
%%time

!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'fine_tuning' / 'bert_experiments.py'} \
  --model-path /kaggle/working/bert_base_ce_model \
  --data-path {DATA_ROOT / 'few_nerd_mountains_output'} \
  --evaluation-split test \
  --device cuda \
  --disable-wandb

### wce Bert evaluation

In [ ]:
%%time

!PYTHONPATH={PROJECT_DIR}:{SRC_DIR} python {NER_DIR / 'fine_tuning' / 'bert_experiments.py'} \
  --model-path /kaggle/working/bert_base_wce_model \
  --data-path {DATA_ROOT / 'few_nerd_mountains_output'} \
  --evaluation-split test \
  --device cuda \
  --disable-wandb

In [ ]:
!zip -r ce_checkpoints.zip /kaggle/working/bert_base_ce_model
!zip -r ce_results.zip /kaggle/working/bert_base_ce_results


!zip -r wce_checkpoints.zip /kaggle/working/bert_base_wce_model
!zip -r wce_results.zip /kaggle/working/bert_base_wce_results

In [ ]:
## some Kaggle issues 
import os
from pathlib import Path
import wandb

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "mountain-ner-upload")

ARCHIVES = [
    "/kaggle/working/ce_checkpoints.zip",
    "/kaggle/working/ce_results.zip",
    "/kaggle/working/wce_checkpoints.zip",
    "/kaggle/working/wce_results.zip",
]

if not WANDB_API_KEY or not WANDB_ENTITY or not WANDB_PROJECT:
    raise ValueError("Set WANDB_API_KEY, WANDB_ENTITY, and WANDB_PROJECT in the environment before uploading.")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)

with wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    job_type="upload-artifacts",
) as run:
    artifact = wandb.Artifact(
        name="mountain-ner-bert-archives",
        type="model",
        metadata={
            "task": "mountain-ner",
            "files": [Path(x).name for x in ARCHIVES],
        },
    )

    for file_path in ARCHIVES:
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"Missing archive: {path}")
        artifact.add_file(local_path=str(path), name=path.name)
        print("added:", path.name)

    run.log_artifact(artifact)
    artifact.wait()
    print("uploaded artifact:", artifact.name)


In [ ]:
# Create a report-ready quality versus latency figure from the recorded test metrics.
from IPython.display import Image, display

TRADEOFF_PLOT = WORK_DIR / 'ner_model_tradeoff.png'
!python {NER_DIR / 'utils' / 'plot_ner_results.py'} --output "{TRADEOFF_PLOT}"
display(Image(filename=str(TRADEOFF_PLOT)))